# RAPID hydrograph lag investigation

This notebook is a focused review tool for states whose outlet lag relative to the inflow peak changes after collapse. It is intentionally not a correlation or predictor-selection notebook.


## Scope and reading order

The notebook builds one validated state-level analysis table, then separates timing responses into three mutually exclusive sections:

1. **Zero lag shift** — timing ties; inspect their selected-domain equivalent-resistance distribution.
2. **Positive lag shift** — delayed outlet peaks; inspect lag and equivalent-resistance distributions.
3. **Negative lag shift** — earlier outlet peaks; review separately rather than mixing them with the other response classes.

Each section uses a stable index and a compact table. Change its `*_REVIEW_INDEX` to inspect another state in the shared map/hydrograph view. `Lambda_matched` is the selected-domain resistance ratio, \(K_{after}/K_{before}\).


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "RAPID" / "src" / "rapid_tools").exists():
            return candidate
    raise FileNotFoundError("Could not locate the river-hierarchy repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / "RAPID" / "src"
OUTPUT_ROOT = REPO_ROOT / "network_variants" / "output"
if str(RAPID_SRC) not in sys.path:
    sys.path.insert(0, str(RAPID_SRC))

from rapid_tools import (
    build_geometry_context_frame,
    build_multi_example_timing_frame,
    make_timing_scatter,
    supported_timing_x_metrics,
    supported_timing_y_metrics,
)

## Configuration and input assembly

The cells below load the completed experiments and form the single analysis population used by every investigation section. Run them before using the review cells.


In [ ]:
EXAMPLE_IDS = ["02", "03", "05","11", "13","15", "21", '56']

# matched_parent_vs_network: fraction of the full network represented by the matched parent corridor
# child_vs_matched_parent: fraction of the matched parent corridor retained by the child object
LENGTH_MODE = "matched_parent_vs_network"

# For complete closed compounds, the exposure factor is 1 because Lambda_matched
# already represents the whole compound; only nested/unit domains use routed width share.
PRIMARY_X_METRIC = "len_fraction_width_weighted_log_lambda"
# Main result: peak-lag shift normalized by the observed upstream-to-downstream gauge travel time.
PRIMARY_Y_METRIC = "lag_to_inflow_peak_pct_shift_vs_base"

# Optional sensitivity filter. Keep False for the complete result set.
EXCLUDE_NETWORK_EDGE_COMPOUNDS = True

# IMPORTANT — CHECK MATCHED-BOUNDARY VALIDITY FOR ALL NETWORKS BEFORE INTERPRETING LAMBDA-MATCHED RESULTS.
# EXCLUDE OR SEPARATELY FLAG ROWS WITH boundary_relation != 'exact_match': THEY ARE NOT STRICT WHOLE-OBJECT RESISTANCE COMPARISONS.

## Build the validated analysis population

Only states with the required RAPID and selected-domain equivalent-resistance records enter `plot_df`. The following cell is the shared input contract for all three response classes.


In [ ]:
timing_df = build_multi_example_timing_frame(EXAMPLE_IDS, output_root=OUTPUT_ROOT)
analysis_df = build_geometry_context_frame(timing_df, length_mode=LENGTH_MODE)

for column in [PRIMARY_X_METRIC, PRIMARY_Y_METRIC, "Lambda_matched"]:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")

analysis_df["primary_pair_complete"] = (
    np.isfinite(analysis_df[PRIMARY_X_METRIC])
    & np.isfinite(analysis_df[PRIMARY_Y_METRIC])
)

coverage = (
    analysis_df.groupby(["example_id", "mode"], dropna=False)
    .agg(
        states=("state_id", "nunique"),
        resistance_available=("Lambda_matched", lambda s: np.isfinite(s).sum()),
        timing_available=(PRIMARY_Y_METRIC, lambda s: np.isfinite(s).sum()),
        primary_pairs=("primary_pair_complete", "sum"),
    )
    .reset_index()
)
display(coverage)
display(analysis_df["plot_category"].value_counts(dropna=False).rename("states").to_frame())

## Apply the population filter

This applies the configured edge-compound filter and produces `plot_df`, the common state-level table used below. No association screening or model-selection figures are retained here.


In [ ]:
plot_df = analysis_df.copy()

if EXCLUDE_NETWORK_EDGE_COMPOUNDS:
    max_order = plot_df.groupby("example_id")["compound_downstream_order"].transform("max")
    is_edge_compound = (
        plot_df["compound_downstream_order"].notna()
        & ((plot_df["compound_downstream_order"] <= 1) | (plot_df["compound_downstream_order"] >= max_order))
    )
    plot_df = plot_df.loc[~is_edge_compound].copy()

plot_df = plot_df.loc[plot_df["primary_pair_complete"]].copy()
print(f"Plotting {len(plot_df)} states from {plot_df['example_id'].nunique()} examples.")

## Shared review helpers

These helpers define the three lag classes, compact distributions, stable review indices, and the reusable state inspector.

In [ ]:
import plotly.express as px

# Shared state-level review frame. Timing classes are mutually exclusive and use
# the stored percentage shift relative to the base state.
investigation_df = plot_df.copy()
lag_metric = "lag_to_inflow_peak_pct_shift_vs_base"
EQ_METRIC = "Lambda_matched"
investigation_df[lag_metric] = pd.to_numeric(investigation_df[lag_metric], errors="coerce")
investigation_df[EQ_METRIC] = pd.to_numeric(investigation_df[EQ_METRIC], errors="coerce")
investigation_df["neg_log10_lambda"] = -np.log10(investigation_df[EQ_METRIC].where(lambda s: s > 0))

compact_columns = [
    "review_index", "example_id", "mode", "state_id", "target_id", "plot_category",
    lag_metric, EQ_METRIC, "neg_log10_lambda", "hydraulic_domain_type",
    "parent_domain_n_links", "child_domain_n_links", "length_fraction",
    "width_fraction_of_total",
]

def _lag_subset_index_plot(frame, *, title, index_value):
    fig = px.scatter(
        frame, x="review_index", y=lag_metric, color="mode", symbol="plot_category",
        hover_data=[c for c in compact_columns if c in frame.columns and c != "review_index"],
        template="plotly_white", title=title,
    )
    fig.update_traces(marker=dict(size=10, line=dict(width=0.7, color="black")))
    fig.add_hline(y=0, line_dash="dash", line_color="grey")
    fig.add_vline(x=index_value, line_dash="dot", line_color="black")
    fig.update_xaxes(dtick=1, title="subset index (change the REVIEW_INDEX below)")
    fig.update_yaxes(title="lag to inflow peak shift vs base [%]")
    return fig

def _eq_distribution(frame, *, title):
    fig = px.histogram(
        frame, x=EQ_METRIC, color="mode", facet_col="plot_category",
        barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title=title,
        hover_data=[c for c in ["example_id", "state_id", "target_id", lag_metric] if c in frame],
    )
    fig.update_xaxes(type="log", title="Lambda_K (selected hydraulic domain; log scale)")
    fig.update_yaxes(title="state count")
    return fig


## Shared state inspector

The inspector is called by each section below. It shows the selected hydraulic domain when available; otherwise it explicitly falls back to the full-network geometry view.

In [ ]:
# Shared visual inspector used by all three response sections. It maps the selected state, overlays base and collapsed outlet hydrographs, and displays only compact calculation/domain tables.
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _investigation_experiment_dir(example_id, mode, state_id=None):
    """Resolve by actual state location; plot labels are not output-directory keys."""
    preferred_suffix = "indep__anchored" if mode == "indep" else "compound_bubbles__anchored"
    preferred = OUTPUT_ROOT / f"{int(example_id):02d}_{preferred_suffix}"
    candidates = [preferred]
    candidates.extend(sorted(OUTPUT_ROOT.glob(f"{int(example_id):02d}_*__anchored")))
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if state_id is None or (candidate / "states" / str(state_id)).exists():
            return candidate
    raise FileNotFoundError(f"No output experiment contains example {example_id}, state {state_id}.")

def _directed_links_path(state_dir):
    paths = [path for path in sorted((state_dir / "variant" / "directed").glob("*_directed_links.gpkg")) if not path.name.startswith("._")]
    if not paths:
        raise FileNotFoundError(f"No directed links found for {state_dir}")
    return paths[0]

def _add_link_layer(fig, links, *, name, color, width, opacity=1.0, row=1, col=1):
    xs, ys = [], []
    for geometry in links.geometry.dropna():
        geometries = list(geometry.geoms) if geometry.geom_type == "MultiLineString" else [geometry]
        for line in geometries:
            x, y = line.xy
            xs.extend(x); ys.extend(y); xs.append(None); ys.append(None)
    if xs:
        fig.add_trace(go.Scattergl(x=xs, y=ys, mode="lines", line=dict(color=color, width=width), opacity=opacity, name=name, hoverinfo="skip"), row=row, col=col)

def inspect_collapsed_state(example_id, mode, state_id):
    experiment_dir = _investigation_experiment_dir(example_id, mode, state_id)
    base_dir = experiment_dir / "states" / "S000_base"
    state_dir = experiment_dir / "states" / state_id
    transition = pd.read_csv(experiment_dir / "equivalent_transition_registry.csv")
    transition_matches = transition.loc[transition["child_state_id"].astype(str).eq(state_id)]
    transition_row = transition_matches.iloc[0] if not transition_matches.empty else None
    domain_path = state_dir / "equivalent_resistance" / "selected_hydraulic_domain_links.csv"
    base_links = gpd.read_file(_directed_links_path(base_dir))
    child_links = gpd.read_file(_directed_links_path(state_dir))
    if domain_path.exists():
        domain = pd.read_csv(domain_path)
        parent_ids = pd.to_numeric(domain.loc[domain["domain_role"].astype(str).eq("parent"), "id_link"], errors="coerce").dropna().astype(int)
        child_ids = pd.to_numeric(domain.loc[domain["domain_role"].astype(str).eq("child"), "id_link"], errors="coerce").dropna().astype(int)
        parent_domain = base_links.loc[pd.to_numeric(base_links["id_link"], errors="coerce").isin(parent_ids)]
        child_domain = child_links.loc[pd.to_numeric(child_links["id_link"], errors="coerce").isin(child_ids)]
        domain_table_columns = ["domain_role", "state_id", "id_link", "id_us_node", "id_ds_node", "len", "wid_adj_routing", "wid_adj_wet", "wid_adj_total"]
        domain_table = domain[[column for column in domain_table_columns if column in domain]].copy()
        domain_note = None
    else:
        # Some plotted states were not written to the selected-domain EQ archive.
        # Keep the visual diagnostic usable, but do not imply an EQ calculation.
        parent_domain, child_domain = base_links, child_links
        domain_table = pd.DataFrame({"inspection note": ["Selected-domain EQ snapshot unavailable; full base and child networks are shown."]})
        domain_note = domain_table.iloc[0, 0]
    base_hydro = pd.read_csv(base_dir / "rapid" / "run" / "outlet_hydrograph.csv")
    child_hydro = pd.read_csv(state_dir / "rapid" / "run" / "outlet_hydrograph.csv")

    fig = make_subplots(rows=1, cols=2, subplot_titles=("Selected hydraulic domain", "Outlet hydrograph"), column_widths=[0.58, 0.42])
    _add_link_layer(fig, base_links, name="base network", color="#9aa0a6", width=1.0, opacity=0.35)
    _add_link_layer(fig, parent_domain, name="base selected domain", color="#1f77b4", width=4.0)
    _add_link_layer(fig, child_links, name="collapsed network", color="#ffbf69", width=1.0, opacity=0.25)
    _add_link_layer(fig, child_domain, name="collapsed selected domain", color="#d62728", width=4.0)
    # State-local time_seconds includes each run's spin-up origin. Plot on the
    # shared UTC forcing clock so apparent offsets are physical peak shifts.
    base_hydro["time_utc"] = pd.to_datetime(base_hydro["time_utc"], utc=True)
    child_hydro["time_utc"] = pd.to_datetime(child_hydro["time_utc"], utc=True)
    fig.add_trace(go.Scatter(x=base_hydro["time_utc"], y=base_hydro["q_outlet_cms"], mode="lines", line=dict(color="#1f77b4", width=2.5), name="base outlet"), row=1, col=2)
    fig.add_trace(go.Scatter(x=child_hydro["time_utc"], y=child_hydro["q_outlet_cms"], mode="lines", line=dict(color="#d62728", width=2.5), name="collapsed outlet"), row=1, col=2)
    fig.update_xaxes(title_text="x [m]", scaleanchor="y", scaleratio=1, row=1, col=1)
    fig.update_yaxes(title_text="y [m]", row=1, col=1)
    fig.update_xaxes(title_text="time [UTC]", row=1, col=2)
    fig.update_yaxes(title_text="discharge [m³ s⁻¹]", row=1, col=2)
    fig.update_layout(
        title=(
            f"Example {int(example_id):02d}, {state_id} — selected-domain resistance and RAPID response"
            if transition_row is None else
            f"Example {int(example_id):02d}, {state_id} — selected-domain resistance and RAPID response"
            f"<br><sup>domain={transition_row.hydraulic_domain_type}; K_before={transition_row.K_before:.4g}; "
            f"K_after={transition_row.K_after:.4g}; Lambda_K={transition_row.Lambda_K:.6g}</sup>"
        ),
        template="plotly_white", width=1550, height=720, legend=dict(orientation="h"), hovermode="closest",
    )
    fig.show()
    # Keep the tables compact. Some analysis rows have no transition-registry match
    # (for example, a state omitted from the selected-domain EQ registry), but their
    # saved domain snapshot, geometry, and RAPID output can still be inspected.
    if transition_row is None or domain_note is not None:
        message = domain_note or "No matching selected-domain EQ transition row; showing saved domain and RAPID outputs only."
        display(pd.DataFrame({"inspection note": [message]}))
    else:
        display(transition_row[[column for column in ["hydraulic_domain_type", "selected_unit_ids", "parent_n_member_units", "parent_domain_n_links", "child_domain_n_links", "K_before", "K_after", "Lambda_K", "G_before", "G_after", "Q_unit_before", "Q_unit_after", "parent_solver_method", "child_solver_method", "residual_relative_before", "residual_relative_after"] if column in transition_row.index]].to_frame("calculation"))
    display(domain_table)

# The lag-review sections call inspect_collapsed_state for their selected index.

## 1. Zero lag shift — timing ties

These states have exactly zero stored shift in lag to the inflow peak. The purpose is to assess their selected-domain equivalent-resistance distribution and inspect individual ties, not to treat them as delayed responses.

In [ ]:
ZERO_LAG_REVIEW_INDEX = 2
zero_lag_cases = investigation_df.loc[investigation_df[lag_metric].eq(0)].sort_values(
    ["example_id", "mode", "state_id"]
).reset_index(drop=True)
zero_lag_cases.insert(0, "review_index", zero_lag_cases.index)
if zero_lag_cases.empty:
    raise ValueError("No zero lag-shift states were found.")
if not 0 <= ZERO_LAG_REVIEW_INDEX < len(zero_lag_cases):
    raise IndexError(f"ZERO_LAG_REVIEW_INDEX must be between 0 and {len(zero_lag_cases) - 1}")
zero_lag_selected = zero_lag_cases.iloc[ZERO_LAG_REVIEW_INDEX]

display(pd.DataFrame({"zero_lag_states": [len(zero_lag_cases)], "all_positive_or_zero": [int(investigation_df[lag_metric].ge(0).sum())]}))
_eq_distribution(zero_lag_cases, title="Zero lag shift: selected-domain Lambda_K distribution").show()
_lag_subset_index_plot(zero_lag_cases, title=f"Zero lag shift candidates (n={len(zero_lag_cases)})", index_value=ZERO_LAG_REVIEW_INDEX).show()
display(zero_lag_selected[[c for c in compact_columns if c in zero_lag_selected.index]].to_frame("selected zero-lag state"))
inspect_collapsed_state(str(zero_lag_selected.example_id), str(zero_lag_selected.mode), str(zero_lag_selected.state_id))


## 2. Positive lag shift — delayed outlet peak

These states have a positive relative lag shift. This section shows the lag distribution and selected-domain resistance-ratio distribution, then lets you inspect one state at a time.

In [ ]:
POSITIVE_LAG_REVIEW_INDEX = 0
positive_lag_cases = investigation_df.loc[investigation_df[lag_metric].gt(0)].sort_values(
    [lag_metric, "example_id", "mode", "state_id"]
).reset_index(drop=True)
positive_lag_cases.insert(0, "review_index", positive_lag_cases.index)
if positive_lag_cases.empty:
    raise ValueError("No positive lag-shift states were found.")
if not 0 <= POSITIVE_LAG_REVIEW_INDEX < len(positive_lag_cases):
    raise IndexError(f"POSITIVE_LAG_REVIEW_INDEX must be between 0 and {len(positive_lag_cases) - 1}")
positive_lag_selected = positive_lag_cases.iloc[POSITIVE_LAG_REVIEW_INDEX]

display(pd.DataFrame({"positive_lag_states": [len(positive_lag_cases)], "lag_min_pct": [positive_lag_cases[lag_metric].min()], "lag_max_pct": [positive_lag_cases[lag_metric].max()]}))
px.histogram(positive_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Positive lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
_eq_distribution(positive_lag_cases, title="Positive lag shift: selected-domain Lambda_K distribution").show()
_lag_subset_index_plot(positive_lag_cases, title=f"Positive lag shift candidates (n={len(positive_lag_cases)})", index_value=POSITIVE_LAG_REVIEW_INDEX).show()
display(positive_lag_selected[[c for c in compact_columns if c in positive_lag_selected.index]].to_frame("selected positive-lag state"))
inspect_collapsed_state(str(positive_lag_selected.example_id), str(positive_lag_selected.mode), str(positive_lag_selected.state_id))


## 3. Negative lag shift — earlier outlet peak

These states have a negative relative lag shift. They are kept separate because their physical interpretation differs from delayed peak timing; use the same distributions and inspector to review them.

In [ ]:
NEGATIVE_LAG_REVIEW_INDEX = 0
negative_lag_cases = investigation_df.loc[investigation_df[lag_metric].lt(0)].sort_values(
    [lag_metric, "example_id", "mode", "state_id"]
).reset_index(drop=True)
negative_lag_cases.insert(0, "review_index", negative_lag_cases.index)
if negative_lag_cases.empty:
    raise ValueError("No negative lag-shift states were found.")
if not 0 <= NEGATIVE_LAG_REVIEW_INDEX < len(negative_lag_cases):
    raise IndexError(f"NEGATIVE_LAG_REVIEW_INDEX must be between 0 and {len(negative_lag_cases) - 1}")
negative_lag_selected = negative_lag_cases.iloc[NEGATIVE_LAG_REVIEW_INDEX]

display(pd.DataFrame({"negative_lag_states": [len(negative_lag_cases)], "lag_min_pct": [negative_lag_cases[lag_metric].min()], "lag_max_pct": [negative_lag_cases[lag_metric].max()]}))
px.histogram(negative_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Negative lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
_eq_distribution(negative_lag_cases, title="Negative lag shift: selected-domain Lambda_K distribution").show()
_lag_subset_index_plot(negative_lag_cases, title=f"Negative lag shift candidates (n={len(negative_lag_cases)})", index_value=NEGATIVE_LAG_REVIEW_INDEX).show()
display(negative_lag_selected[[c for c in compact_columns if c in negative_lag_selected.index]].to_frame("selected negative-lag state"))
inspect_collapsed_state(str(negative_lag_selected.example_id), str(negative_lag_selected.mode), str(negative_lag_selected.state_id))
